In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from preprocessing import get_features_and_target
from visualizer import plot_visualizer
import plotly.graph_objects as go
from tabpfn import TabPFNRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor

In [2]:
import huggingface_hub
huggingface_hub.login()

# Getting Dataframe

In [ ]:
# Load the training and development datasets
train_df = pd.read_csv("data/train_data.csv")
dev_df = pd.read_csv("data/development_data.csv")
sc = StandardScaler()

target_column = "PullTest (N)" 

x_train, y_train = get_features_and_target(train_df, target_column)
x_dev, y_dev = get_features_and_target(dev_df, target_column)

x_train_scale = sc.fit_transform(X=x_train)
x_dev_scale = sc.transform(X=x_dev)

# Defining Model

In [ ]:
#model = 'XGBoost'
#model = 'RandomForest'
model = 'TabPFN'

# Physical Calculation

In [4]:
def compute_interfacial_failure(df):
     t = df[['Thickness A (mm)', 'Thickness B (mm)']].min(axis=1) 
     f_pull = 1 * (np.pi/4) * (4 * np.sqrt(t))**2 * (0.7 * 365) 
     return np.round(f_pull, 1) 

def compute_pullout_failure(df):
     t = df[['Thickness A (mm)', 'Thickness B (mm)']].min(axis=1) 
     f_pull = np.pi * ((4 * np.sqrt(t)) + 2*t)*t*365 
     return np.round(f_pull, 1) 

x_train['Interfacial_Failure'] = compute_interfacial_failure(x_train) 
x_train['Pullout_Failure'] = compute_pullout_failure(x_train) 
x_dev['Interfacial_Failure'] = compute_interfacial_failure(x_dev)
x_dev['Pullout_Failure'] = compute_pullout_failure(x_dev)

y_train_delta = y_train - x_train['Pullout_Failure']
y_dev_delta = y_dev - x_dev['Pullout_Failure']

In [5]:
print(y_dev)

0     4161.4
1     1836.4
2     2509.8
3     2867.4
4     5277.7
       ...  
88    2937.9
89    3028.8
90    2860.7
91    2816.5
92    2978.6
Name: PullTest (N), Length: 93, dtype: float64


# Fit Model

In [ ]:
if model == 'TabPFNRegressor':   
    # Initialize the regressor
    regressor = TabPFNRegressor()  # Uses TabPFN-2.5 weights, trained on synthetic data only.
    # To use TabPFN v2:
    # regressor = TabPFNRegressor.create_default_for_version(ModelVersion.V2)
    regressor.fit(x_train, y_train_delta)

    # Predict on the test set
    predictions = regressor.predict(x_dev)
    final_pred = x_dev['Pullout_Failure'] + predictions
elif model == 'XGBoost':

    # Convert the data into DMatrix format
    dtrain = xgb.DMatrix(x_train_scale, label=y_train)
    dtest = xgb.DMatrix(x_dev_scale, label=y_dev)

    # Set the parameters for the XGBoost model
    params = {
        'objective': 'reg:squarederror',
        'max_depth': 5,
        'eta': 0.3,
        'eval_metric': 'rmse'
    }

    # Train the model
    num_boost_round = 10
    bst = xgb.train(params, dtrain, num_boost_round)

    # Make predictions
    predictions = bst.predict(dtest)

# Check Validation Data

In [ ]:
# Category array (must be aligned with y_dev)
categories = dev_df.groupby("Sample ID")["Category"].first().values

plot_visualizer(
    true_vals=y_dev,
    pred_vals=final_pred,
    categories=categories,
    title=f"Validation Samples: True vs Prediction ({model}) by Category"
)

# Check Validation Loss and R2

In [8]:
# Calculate MAE and RMSE and R2
mae  = mean_absolute_error(y_dev, final_pred)
rmse = root_mean_squared_error(y_dev, final_pred)
R2   = r2_score(y_dev, final_pred)


print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2: {R2:.2f}")


MAE:  116.50
RMSE: 194.42
R2: 0.69


# Cross Validation

In [ ]:
cross_df = pd.read_csv("data/train_dev_data.csv")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

mae_list = []
rmse_list = []
R2_list = []

for fold, (train_index, val_index) in enumerate(skf.split(cross_df["Sample ID"], cross_df["Category"])):
    x_tr, y_tr = get_features_and_target(cross_df.iloc[train_index], target_column) 
    x_val, y_val = get_features_and_target(cross_df.iloc[val_index], target_column)

    x_tr['Interfacial_Failure'] = compute_interfacial_failure(x_tr) 
    x_tr['Pullout_Failure'] = compute_pullout_failure(x_tr) 
    x_val['Interfacial_Failure'] = compute_interfacial_failure(x_val)
    x_val['Pullout_Failure'] = compute_pullout_failure(x_val)

    y_train_delta = y_tr - x_tr['Pullout_Failure'] 
    y_dev_delta = y_val - x_val['Pullout_Failure']

    regressor = TabPFNRegressor()
    regressor.fit(x_tr, y_train_delta)

    preds = regressor.predict(x_val)
    final_pred = x_val['Pullout_Failure'] + preds

    mae  = mean_absolute_error(y_val, final_pred)
    rmse = root_mean_squared_error(y_val, final_pred)
    R2   = r2_score(y_val, final_pred)

    mae_list.append(mae) 
    rmse_list.append(rmse) 
    R2_list.append(R2)

    # Categories
    categories= cross_df.iloc[val_index]["Category"].values

    plot_visualizer(
        true_vals=y_val,
        pred_vals=final_pred,
        categories=categories,
        title=f"Fold {fold+1}: True vs Prediction ({model}) by Category"
    )

    print(f"\nFold {fold+1}")
    print("MAE :", mae)
    print("RMSE:", rmse)
    print("R²  :", R2)

mae_mean = np.mean(mae_list) 
rmse_mean = np.mean(rmse_list) 
R2_mean = np.mean(R2_list) 



Fold 1
MAE : 124.99077664968125
RMSE: 190.08671085926335
R²  : 0.6211203993594667



Fold 2
MAE : 141.64038861248946
RMSE: 284.4481413853948
R²  : 0.6993303232246222



Fold 3
MAE : 120.73128320204243
RMSE: 200.04512725992126
R²  : 0.803842749287466



Fold 4
MAE : 109.73493524145437
RMSE: 146.66155103389548
R²  : 0.7232278912182819



Fold 5
MAE : 108.32015319197146
RMSE: 176.9656274071561
R²  : 0.647897260348518


In [12]:
# Final Evaluation over all Folds
print(f"mean MAE:  {mae_mean:.2f}")
print(f"mean RMSE: {rmse_mean:.2f}")
print(f"mean R²:   {R2_mean:.2f}")

mean MAE:  121.08
mean RMSE: 199.64
mean R²:   0.70
